# 소희 알약 이미지 EDA

원본을 수정하지 않고 JSON 연결 → bbox 확인 → 무결성 → 분포 → 촬영 조건 → 분할 후보를 분석합니다.
실행 환경: 프로젝트의 uv 환경. 커널의 작업 폴더는 프로젝트 루트 또는 notebooks/를 지원합니다.
전체 JSON은 메타데이터만 읽고, 이미지는 한 장씩 읽어 메모리 사용을 제한합니다.
노트북 출력은 저장하지 않으며 요약 보고서와 PNG는 실행 시 생성됩니다. seed=42.
기존 data_preprocessing.ipynb 및 dongwook_eda.ipynb는 변경하지 않습니다.


In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ["WANDB_MODE"] = "disabled"
import pandas as pd
from IPython.display import Image, display

root = Path.cwd().resolve()
if not (root / "src" / "config.py").is_file():
    root = root.parent
if not (root / "src" / "config.py").is_file():
    raise RuntimeError("프로젝트 루트 또는 notebooks 폴더에서 실행하세요.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.config import PROJECT_ROOT
from notebooks.sohee_eda_utils import (
    CONDITIONS, draw_boxes, image_paths, inspect_images, load_annotations,
    near_duplicate_candidates, plot_distributions, prepare_tables, suggest_split, write_report,
)

SEED = 42
output_dir = PROJECT_ROOT / "notebooks" / "sohee_eda_figures"
output_dir.mkdir(exist_ok=True)
train_paths = image_paths(PROJECT_ROOT / "data" / "train_images")
test_paths = image_paths(PROJECT_ROOT / "data" / "test_images")
print({"train_images": len(train_paths), "test_images": len(test_paths), "seed": SEED})


## 1. 샘플 구조 확인
처음 세 상위 폴더에서 JSON 하나씩 읽어 자료형·이미지 ID·클래스·bbox를 확인합니다.
전역 image_id로 결합하지 않습니다. 전체 분석에서도 JSON 내부 참조를 먼저 확인합니다.


In [ ]:
annotation_dir = PROJECT_ROOT / "data" / "train_annotations"
sample_rows = []
for folder in sorted(annotation_dir.iterdir())[:3]:
    if not folder.is_dir():
        continue
    sample = next(iter(sorted(folder.rglob("*.json"))), None)
    if sample is None:
        continue
    payload = json.loads(sample.read_text(encoding="utf-8-sig"))
    sample_rows.append({
        "source": str(sample.relative_to(PROJECT_ROOT)),
        "types": {key: type(payload[key]).__name__ for key in ["images", "annotations", "categories"]},
        "images": payload["images"], "annotations": payload["annotations"],
        "categories": payload["categories"],
    })
display(pd.DataFrame(sample_rows))


## 2. 이미지 단위 연결 기준 확정
전체 JSON 메타데이터를 순차적으로 읽습니다. 파일명을 고유 이미지와 대응시키고 출처를 유지합니다.
동일 이미지·클래스·bbox 중복은 표시만 하며, 분포용 테이블에서 제외합니다.
객체가 없는 이미지가 실제 음성 이미지인지 라벨 누락인지는 자동 판단하지 않습니다.


In [ ]:
records, json_files, load_issues = load_annotations(annotation_dir)
records, image_lookup = prepare_tables(records, train_paths)
objects = records.loc[records.image_present & ~records.duplicate].copy()
if objects.empty:
    raise RuntimeError("이미지와 연결된 객체가 없습니다. 경로와 파일명을 확인하세요.")
print({"json": len(json_files), "records": len(records), "linked_objects": len(objects)})
display(records.groupby("image_id").file_name.nunique().sort_values(ascending=False).head())
display(records.groupby("file_name").agg(json_count=("source", "nunique"),
                                         image_ids=("image_id", "nunique"),
                                         classes=("category_id", "nunique")).head())
display(load_issues.head(10))


## 3. bbox 시각 검증
서로 다른 알약 조합의 이미지 최대 세 장을 선택하고 모든 연결된 bbox와 제품명을 표시합니다.
xywh 해석, 이미지 회전, 박스 위치와 제품명을 사람이 확인해야 합니다.


In [ ]:
sample_names = (objects[["file_name", "capture_group"]].drop_duplicates()
                .drop_duplicates("capture_group").head(3).file_name.tolist())
draw_boxes(records, sample_names, output_dir / "bbox_samples.png")
display(Image(filename=str(output_dir / "bbox_samples.png")))


## 4. 기본 규모와 무결성 분석
모든 학습·테스트 이미지를 한 장씩 디코딩하고 SHA256·해상도·품질 지표를 계산합니다.
배경을 포함한 Laplacian 분산은 흐림 확정값이 아닙니다. 테스트 클래스는 추정하지 않습니다.


In [ ]:
train_quality = inspect_images(train_paths)
test_quality = inspect_images(test_paths)
annotated_names = set(records.file_name)
actual_names = set(train_quality.file_name)
display(pd.Series({
    "annotation_images": len(annotated_names),
    "annotation_without_image": len(annotated_names - actual_names),
    "image_without_annotation": len(actual_names - annotated_names),
    "invalid_bbox": int((~records.bbox_valid).sum()),
    "area_mismatch": int((~records.area_matches).sum()),
    "duplicate_records": int(records.duplicate.sum()),
    "train_decode_failures": int((~train_quality.readable).sum()),
    "test_decode_failures": int((~test_quality.readable).sum()),
}))
display(records.loc[~records.bbox_valid | ~records.area_matches,
                    ["source", "file_name", "category_id", "bbox_valid", "area_matches"]].head(20))
display(train_quality.groupby(["actual_width", "actual_height"]).size())


## 5. 클래스·객체·bbox 분포 분석
전체 annotation과 로컬 이미지 연결 객체를 구분합니다. 아래 그래프는 연결된 객체 기준입니다.
클래스별 이미지 수와 객체 수를 별도로 계산하고, 좌표 통계에는 유효한 bbox만 사용합니다.


In [ ]:
class_counts = objects.groupby(["category_id", "category_name"]).agg(
    objects=("file_name", "size"), images=("file_name", "nunique"),
    groups=("capture_group", "nunique"),
).sort_values("objects", ascending=False)
display(class_counts)
display(records.groupby("category_id").size().describe().rename("전체 annotation 클래스당 객체 수"))
display(objects.groupby("file_name").size().describe().rename("연결된 이미지당 객체 수"))
plot_distributions(objects.loc[objects.bbox_valid], train_quality, output_dir / "distributions.png")
display(Image(filename=str(output_dir / "distributions.png")))


## 6. 촬영 조건과 어려운 사례
촬영 조건별 이미지 수와 클래스 교차표를 확인합니다. 앞뒷면은 제품별 값일 수 있습니다.
흐림 후보를 시각 검토하고, 평균 해시 거리 3 이하의 쌍을 유사 이미지 후보로만 취급합니다.


In [ ]:
condition_tables = {}
for condition in CONDITIONS:
    values = objects[["file_name", condition]].drop_duplicates()
    print(condition)
    display(values[condition].fillna("<결측>").replace("", "<빈 문자열>").value_counts())
    condition_tables[condition] = pd.crosstab(objects.category_id, objects[condition])
    display(condition_tables[condition].head(10))
near_pairs = near_duplicate_candidates(train_quality)
print("평균 해시 유사 쌍 후보:", len(near_pairs))
display(near_pairs.head(10))
readable_names = set(train_quality.loc[train_quality.readable, "file_name"])
linked_quality = train_quality.loc[train_quality.file_name.isin(set(objects.file_name) & readable_names)]
difficult_names = linked_quality.nsmallest(3, "blur_score").file_name.tolist()
draw_boxes(records, difficult_names, output_dir / "difficult_samples.png")
display(Image(filename=str(output_dir / "difficult_samples.png")))


## 7. 분할 기준과 학습 준비 사항
파일명 첫 밑줄 앞의 알약 조합을 촬영 그룹의 대리값으로 사용합니다.
seed=42로 그룹의 약 20%를 검증 후보로 선택하고 학습에 없는 검증 클래스를 확인합니다.
정식 세션 ID·동일/유사 이미지 검수 및 클래스 커버리지 조정 전에는 학습에 사용하지 않습니다.
파일 이동·학습·라벨 변환은 하지 않습니다. 원본이 다른 이미지라도 중복이면 그룹을 합쳐야 합니다.


In [ ]:
split_candidates, split_info = suggest_split(objects, seed=SEED)
display(pd.crosstab(split_candidates.capture_group, split_candidates.split))
display(split_info)
report_path = PROJECT_ROOT / "notebooks" / "sohee_eda_results.md"
write_report(records, json_files, load_issues, train_quality, test_quality,
             objects, split_info, near_pairs, report_path)
print("결과 보고서:", report_path.relative_to(PROJECT_ROOT))


## 검수와 다음 작업
- bbox 시각 검증 후 좌표 해석을 확정합니다.
- 누락 이미지·누락 라벨·bbox 오류·클래스 이름 충돌은 자동 수정하지 않습니다.
- 흐림·반사·가림·유사 제품은 후보 이미지를 사람이 검토합니다.
- 희소 클래스는 이미지 수뿐 아니라 독립 촬영 그룹 수로 평가합니다.
- 색상·각인을 보존하는 범위에서 밝기·대비·회전 등의 증강 후보를 검토합니다.
- 노트북 출력은 지운 상태로 유지합니다. 보고서와 그림은 셀을 실행하면 갱신됩니다.
